# NHS.UK Health A-Z: coleta e preparação de um corpus de texto

Equipe: Guilherme Moll

chatbot de triagem médica


| Critério da avaliação | Onde está |
|---|---|
| Justificativa da escolha da base | seção 1 |
| Volume, abrangência, variedade | seção 1 |
| Dicionário de dados | `DATA_DICTIONARY.md` |
| Script de coleta, funcionamento | seção 2 |
| Remoção de ruídos | seção 3 |
| Tokenização, normalização, stopwords e pontuação | seção 4 |
| Stemming e lematização (bônus) | seção 5 |
| Recortes amostrais (bônus) | seção 6 |
| Automatização da coleta (bônus) | seção 7 |

In [1]:
import gzip, json, re, unicodedata, hashlib, time, random
from collections import Counter, defaultdict
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 130)

with gzip.open("data/passages.jsonl.gz", "rt", encoding="utf-8") as fh:
    passages = [json.loads(line) for line in fh]

print(f"passagens : {len(passages):,}")
print(f"páginas   : {len({p['page_id'] for p in passages}):,}")
print(f"caracteres: {sum(len(p['raw_text']) for p in passages):,}")

passagens : 24,159
páginas   : 3,530
caracteres: 12,419,942


## 1. A base de dados


Dados médicos e de medicamentos escritos em linguagem simples para leigos entenderem, já ordenads em prioridade e conectando doenças com sintomas e remédios

In [2]:
frame = pd.DataFrame(passages)

print("Abrangência: as nove subseções do Health A-Z\n")
display(frame["section"].value_counts().rename("passagens").to_frame())

print("\nVariedade: o tipo de página, atribuído pelo publicador")
display(frame["genre"].replace("", "(sem tipo)").value_counts().head(8).rename("passagens").to_frame())

Abrangência: as nove subseções do Health A-Z



,passagens
section,
conditions,9934
medicines,8153
tests-and-treatments,1859
mental-health,1058
symptoms,914
pregnancy,682
baby,612
contraception,594
vaccinations,353



Variedade: o tipo de página, atribuído pelo publicador


,passagens
genre,
Condition,7310
(sem tipo),6831
Medicine,4980
Advice or information,1902
Symptom,980
Treatment,512
Test,480
Contraception,364


In [3]:
print("Rótulo de urgência, lido direto da página:\n")
display(frame["urgency"].value_counts().rename("passagens").to_frame())

rotulado = frame[frame["care_card"].notna()]
print(f"\n{len(rotulado):,} passagens vêm de um care card, "
      f"em {rotulado['page_id'].nunique():,} páginas distintas.")
print("São os dados de treino já rotulados.")

Rótulo de urgência, lido direto da página:



,passagens
urgency,
informational,21059
self_care,1036
urgent,720
non_urgent,716
emergency,628



1,894 passagens vêm de um care card, em 1,112 páginas distintas.
São os dados de treino já rotulados.


In [4]:
# Um exemplo de cada nível, para mostrar o que os rótulos são.
for nivel in ("emergency", "urgent", "non_urgent", "self_care"):
    linha = frame[frame["urgency"] == nivel].iloc[0]
    print(f"--- {nivel.upper()}  ({linha['page_name']})")
    print(f"    {linha['topic_headline'] or linha['passage_headline']}")
    print(f"    {linha['raw_text'][:150].replace(chr(10), ' / ')}")
    print()

--- EMERGENCY  (Sudden infant death syndrome (SIDS))
    Immediate action required: Call 999 if:
    your baby stops breathing / will not wake up / your baby is struggling for breath / your child is stiff, shaking or jerking (this may be a seizure or fit) / y

--- URGENT  (Sudden infant death syndrome (SIDS))
    Urgent advice: Get an urgent GP appointment or contact 111 if:
    you're worried about your baby or think they might be ill

--- NON_URGENT  (Nappy rash)
    When and where to get medical help
    your baby's nappy rash does not go away, gets worse or spreads to other areas / your baby has a high temperature / your baby seems very uncomfortable

--- SELF_CARE  (Nappy rash)
    Things you can do to help with nappy rash
    Nappy rash can be treated and prevented by following some simple advice.



## 2. Coleta

A API oficial foi testada primeiro: `api.nhs.uk` responde
`401 "Access denied due to missing subscription key"`. tentando em HTTP comum, cada página embute um documento schema.org em
`<script type="application/ld+json">`

A descoberta das URLs vem do `sitemap.xml`

In [5]:
import requests
from bs4 import BeautifulSoup

BASE = "https://www.nhs.uk"
SECOES = ("conditions", "symptoms", "tests-and-treatments", "medicines", "baby",
          "contraception", "mental-health", "pregnancy", "vaccinations")


def buscar(url, sessao=None):
    sessao = sessao or requests.Session()
    resposta = sessao.get(url, timeout=30)
    resposta.raise_for_status()
    time.sleep(1 + random.random() * 0.5)
    return resposta


def urls_do_sitemap(sessao=None):
    sessao = sessao or requests.Session()
    indice = BeautifulSoup(buscar(f"{BASE}/sitemap.xml", sessao).text, "xml")
    alvo = next(loc.text for loc in indice.find_all("loc") if "cms-content" in loc.text)

    mapa = BeautifulSoup(buscar(alvo, sessao).text, "xml")
    achados = []
    for no in mapa.find_all("url"):
        url = no.find("loc").text
        secao = url.replace(BASE, "").strip("/").split("/")[0]
        if secao in SECOES:
            lastmod = no.find("lastmod")
            achados.append((url, lastmod.text if lastmod else None))
    return achados

In [6]:
print(frame["extraction_method"].value_counts().rename("passagens").to_frame())

                   passagens
extraction_method           
json_ld                19167
html_fallback           4992


In [7]:
# Teste de uma página.
import extract

ALVO = f"{BASE}/conditions/asthma/"

try:
    html = buscar(ALVO).text
except requests.RequestException as erro:
    print("sem rede agora; o corpus já coletado está em data/passages.jsonl.gz")
    print(type(erro).__name__, erro)
else:
    pagina, novas = extract.extract(html, "conditions__asthma", ALVO, "conditions")
    print(f"{pagina['name']} ({pagina['page_type']})")
    print(f"rota {pagina['extraction_method']}, {len(novas)} passagens\n")
    display(pd.DataFrame(novas)[["health_aspect", "urgency", "raw_text"]].head(6))

Asthma (MedicalWebPage)
rota json_ld, 20 passagens



,health_aspect,urgency,raw_text
0,SymptomsHealthAspect,informational,"The main symptoms of asthma are problems with breathing, such as:\nwheezing\..."
1,MedicalHelpNonurgentHealthAspect,non_urgent,you or your child have asthma symptoms\nyou or your child have asthma and tr...
2,MedicalHelpEmergencyHealthAspect,emergency,You or someone else is having an asthma attack and:\nstart to feel worse at ...
3,MedicalHelpEmergencyHealthAspect,emergency,Do not drive yourself to A&E.\nThe person you speak to at 999 will give you ...
4,MedicalHelpUrgentHealthAspect,urgent,"you've had an asthma attack and have recovered\nEven if you feel better, it'..."
5,DiagnosisHealthAspect,informational,If you or your child have symptoms of asthma a GP or nurse will ask you some...


In [8]:
# Regenera data/passages.jsonl.gz do zero. ~3.500 requisicoes, cerca de 1 hora.
COLETAR_TUDO = False        # deixe False para nao recoletar ao rodar tudo


def page_id_de(url):
    return url.replace(BASE, "").strip("/").replace("/", "__")


if COLETAR_TUDO:
    sessao = requests.Session()
    todas, falhas = [], []

    for url, _ in urls_do_sitemap(sessao):
        secao = url.replace(BASE, "").strip("/").split("/")[0]
        try:
            _, novas = extract.extract(buscar(url, sessao).text,
                                       page_id_de(url), url, secao)
            todas.extend(novas)
        except Exception as erro:
            falhas.append((url, str(erro)))

    with gzip.open("data/passages.jsonl.gz", "wt", encoding="utf-8") as fh:
        for registro in todas:
            fh.write(json.dumps(registro, ensure_ascii=False) + "\n")

    print(f"{len(todas):,} passagens gravadas, {len(falhas)} paginas falharam")
else:
    print("COLETAR_TUDO = False; usando o corpus ja coletado em data/passages.jsonl.gz")

COLETAR_TUDO = False; usando o corpus ja coletado em data/passages.jsonl.gz


## 3. Limpeza: remoção de ruídos




In [9]:
TAG = re.compile(r"<[^>]+>")
ENTIDADE = re.compile(r"&(?:[a-zA-Z]+|#\d+|#x[0-9a-fA-F]+);")
URL = re.compile(r"https?://\S+|www\.\S+")
ESPACOS = re.compile(r"[ \t\xa0]+")
LINHAS = re.compile(r"\n{2,}")

TIPOGRAFICOS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00ad": ""}

PROBLEMAS = {
    "tag HTML residual": TAG,
    "entidade HTML": ENTIDADE,
    "URL no corpo": URL,
    "espaço repetido": re.compile(r"[ \t]{2,}"),
    "espaço inquebrável": re.compile(r"\xa0"),
    "apóstrofo curvo": re.compile(r"[\u2018\u2019]"),
}


def diagnosticar(textos):
    textos = list(textos)
    return {nome: sum(1 for t in textos if padrao.search(t))
            for nome, padrao in PROBLEMAS.items()}


def limpar(texto):
    if not texto:
        return ""
    texto = unicodedata.normalize("NFKC", texto)
    for de, para in TIPOGRAFICOS.items():
        texto = texto.replace(de, para)
    texto = ENTIDADE.sub(" ", TAG.sub(" ", texto))
    texto = URL.sub(" ", texto)                    # a coluna url preserva a referência
    texto = LINHAS.sub("\n", ESPACOS.sub(" ", texto))
    return "\n".join(l.strip() for l in texto.split("\n")).strip()

In [10]:
antes = diagnosticar(p["raw_text"] for p in passages)

for p in passages:
    p["clean_text"] = limpar(p["raw_text"])

depois = diagnosticar(p["clean_text"] for p in passages)

display(pd.DataFrame({"antes": antes, "depois": depois})
        .sort_values("antes", ascending=False))

,antes,depois
apóstrofo curvo,1166,0
URL no corpo,209,0
entidade HTML,0,0
tag HTML residual,0,0
espaço repetido,0,0
espaço inquebrável,0,0


In [11]:
mudou = [p for p in passages if p["raw_text"] != p["clean_text"]]
print(f"a limpeza alterou {len(mudou):,} de {len(passages):,} passagens\n")
exemplo = mudou[0]
print("raw_text  :", repr(exemplo["raw_text"][:140]))
print("clean_text:", repr(exemplo["clean_text"][:140]))

a limpeza alterou 4,755 de 24,159 passagens

raw_text  : 'The checklist in your baby\'s personal child health record ("red book") tells you how to check your child\'s hearing as they grow older.\nTalk '
clean_text: 'The checklist in your baby\'s personal child health record ("red book") tells you how to check your child\'s hearing as they grow older.\nTalk '


remocao de duplicadas

In [12]:
def impressao(texto):
    return hashlib.sha256(" ".join(texto.casefold().split()).encode("utf-8")).hexdigest()


grupos = defaultdict(list)
for p in passages:
    grupos[impressao(p["clean_text"])].append(p)

repetidos = {k: v for k, v in grupos.items() if len(v) > 1}
em_5_paginas = [v for v in repetidos.values() if len({x["page_id"] for x in v}) >= 5]

print(f"textos distintos              : {len(grupos):,}")
print(f"grupos de duplicata exata     : {len(repetidos):,}")
print(f"blocos repetidos em 5+ páginas: {len(em_5_paginas):,}\n")

top = sorted(em_5_paginas, key=lambda v: -len({x['page_id'] for x in v}))[:5]
display(pd.DataFrame([{"paginas": len({x["page_id"] for x in v}),
                       "urgencia": v[0]["urgency"],
                       "trecho": v[0]["clean_text"][:70]} for v in top]))

textos distintos              : 18,031
grupos de duplicata exata     : 3,504
blocos repetidos em 5+ páginas: 221



,paginas,urgencia,trecho
0,183,informational,"Call 999 now if:\nyour lips, mouth, throat or tongue suddenly become sw"
1,177,informational,"Tell your doctor or pharmacist if you're taking any other medicines, i"
2,153,emergency,Do not drive to A&E. Ask someone to drive you or call 999 and ask for
3,150,informational,Tell your doctor if you're:\ntrying to get pregnant\npregnant\nbreastfeed
4,67,informational,Do not drive to A&E. Ask someone to drive you or call 999 and ask for


## 4. Pré-processamento



In [13]:
import nltk
for recurso in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(recurso, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Negação e modalidade
NEGACAO = frozenset({
    "no", "nor", "not", "don", "dont", "ain", "aren", "arent",
    "couldn", "couldnt", "didn", "didnt", "doesn", "doesnt",
    "hadn", "hadnt", "hasn", "hasnt", "haven", "havent", "isn", "isnt",
    "mightn", "mightnt", "mustn", "mustnt", "needn", "neednt",
    "shan", "shant", "shouldn", "shouldnt", "wasn", "wasnt",
    "weren", "werent", "won", "wont", "wouldn", "wouldnt",
    "against", "few", "more", "most", "very", "only", "same",
})
STOPWORDS = frozenset(stopwords.words("english")) - NEGACAO
stemmer = SnowballStemmer("english")


def tokenizar(texto):
    return word_tokenize(texto, language="english")


def normalizar(tokens):
    # isalpha() remove pontuação e numerais.
    return [t.casefold() for t in tokens if t.isalpha()]


def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]


def stems(tokens):
    return [stemmer.stem(t) for t in tokens]

In [14]:
import spacy, subprocess, sys

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


def lemas_em_lote(listas, batch_size=256):
    textos = [" ".join(t) for t in listas]
    return [[t.lemma_ for t in doc] for doc in nlp.pipe(textos, batch_size=batch_size)]

In [15]:
demo = next(p for p in passages
            if p["urgency"] == "emergency" and 90 < len(p["clean_text"]) < 240)

t = tokenizar(demo["clean_text"])
n = normalizar(t)
f = sem_stopwords(n)

print("PÁGINA:", demo["page_name"], "\n")
for rotulo, valor in [("0. clean_text", demo["clean_text"]),
                      ("1. sentenças", sent_tokenize(demo["clean_text"], language="english")),
                      ("2. tokens", t),
                      ("3. normalizados", n),
                      ("4. sem stopwords", f),
                      ("5a. lemas", lemas_em_lote([f])[0]),
                      ("5b. stems", stems(f))]:
    print(f"{rotulo:<18} {valor}\n")

PÁGINA: Acute lymphoblastic leukaemia 

0. clean_text      You or your child:
have a swollen and red face, neck, arms or hands that came on suddenly - on brown and black skin the redness may be harder to see
have swollen veins in the neck or chest that came on suddenly
Find your nearest A&E

1. sentenças       ['You or your child:\nhave a swollen and red face, neck, arms or hands that came on suddenly - on brown and black skin the redness may be harder to see\nhave swollen veins in the neck or chest that came on suddenly\nFind your nearest A&E']

2. tokens          ['You', 'or', 'your', 'child', ':', 'have', 'a', 'swollen', 'and', 'red', 'face', ',', 'neck', ',', 'arms', 'or', 'hands', 'that', 'came', 'on', 'suddenly', '-', 'on', 'brown', 'and', 'black', 'skin', 'the', 'redness', 'may', 'be', 'harder', 'to', 'see', 'have', 'swollen', 'veins', 'in', 'the', 'neck', 'or', 'chest', 'that', 'came', 'on', 'suddenly', 'Find', 'your', 'nearest', 'A', '&', 'E']

3. normalizados    ['you', 'or',

### Negação

As stopwords do NLTK incluem `not`, `no`, `cannot`. Removê-las apaga a diferença
entre "cannot breathe" e "breathe", que é o que a triagem precisa distinguir.

In [16]:
frase = "You cannot breathe and you are not able to speak"
cru = [t for t in normalizar(tokenizar(frase)) if t not in set(stopwords.words("english"))]

print("original     :", frase)
print("sem proteção :", cru)
print("com proteção :", sem_stopwords(normalizar(tokenizar(frase))))
print(f"\n{len(NEGACAO)} palavras de negação e modalidade são protegidas.")

original     : You cannot breathe and you are not able to speak
sem proteção : ['breathe', 'able', 'speak']
com proteção : ['not', 'breathe', 'not', 'able', 'speak']

47 palavras de negação e modalidade são protegidas.


In [17]:
# aplicando sobre o corpus todo
tokens_todos = [tokenizar(p["clean_text"]) for p in passages]
norm_todos = [normalizar(t) for t in tokens_todos]
filt_todos = [sem_stopwords(t) for t in norm_todos]
lemas_todos = lemas_em_lote(filt_todos)
stems_todos = [stems(t) for t in filt_todos]

for p, a, b, c, d, e in zip(passages, tokens_todos, norm_todos, filt_todos,
                            lemas_todos, stems_todos):
    p.update(tokens=a, tokens_normalized=b, tokens_no_stopwords=c, lemmas=d, stems=e)

print(f"{len(passages):,} passagens preparadas")

24,159 passagens preparadas


In [18]:
etapas = ["tokens", "tokens_normalized", "tokens_no_stopwords", "lemmas", "stems"]
vocab = {e: len({t for p in passages for t in p[e]}) for e in etapas}

tabela = pd.DataFrame({"tipos distintos": vocab})
tabela["redução vs tokens"] = (1 - tabela["tipos distintos"] / tabela["tipos distintos"].iloc[0]).map("{:.1%}".format)
display(tabela)

,tipos distintos,redução vs tokens
tokens,20771,0.0%
tokens_normalized,14791,28.8%
tokens_no_stopwords,14673,29.4%
lemmas,11835,43.0%
stems,9712,53.2%


## 5. Stemming e lematização


In [19]:
palavras = ["coughing", "coughed", "breathing", "swelling", "vomiting",
            "bleeding", "medicines", "treatments", "diagnosed", "worsening"]
pd.DataFrame({"token": palavras,
              "lema": lemas_em_lote([palavras])[0],
              "stem": stems(palavras)})

,token,lema,stem
0,coughing,cough,cough
1,coughed,coughed,cough
2,breathing,breathing,breath
3,swelling,swell,swell
4,vomiting,vomiting,vomit
5,bleeding,bleed,bleed
6,medicines,medicine,medicin
7,treatments,treatment,treatment
8,diagnosed,diagnose,diagnos
9,worsening,worsen,worsen


## 6. Recortes amostrais


In [20]:
final = pd.DataFrame(passages)
final["n_palavras"] = final["clean_text"].str.split().str.len()

display(final.groupby("section").agg(
    passagens=("passage_id", "count"),
    paginas=("page_id", "nunique"),
    media_palavras=("n_palavras", "mean"),
).sort_values("passagens", ascending=False).round(1))

,passagens,paginas,media_palavras
section,,,
conditions,9934,1235,71.7
medicines,8153,1410,104.7
tests-and-treatments,1859,310,103.0
mental-health,1058,157,99.5
symptoms,914,108,62.3
pregnancy,682,92,98.2
baby,612,101,118.3
contraception,594,91,86.4
vaccinations,353,26,61.9


In [21]:
def vocabulario(recorte):
    return {t for linha in recorte["lemmas"] for t in linha}


a = vocabulario(final[final["section"] == "conditions"])
b = vocabulario(final[final["section"] == "medicines"])

print(f"lemas em /conditions/ : {len(a):,}")
print(f"lemas em /medicines/  : {len(b):,}")
print(f"compartilhados        : {len(a & b):,}")
print(f"Jaccard               : {len(a & b) / len(a | b):.3f}")
print("\nSobreposição parcial: são registros distintos, não cópias.")

lemas em /conditions/ : 8,435
lemas em /medicines/  : 4,552
compartilhados        : 3,397
Jaccard               : 0.354

Sobreposição parcial: são registros distintos, não cópias.


## 7. Automatização da coleta

O sitemap traz `<lastmod>` por URL.

In [22]:
try:
    achados = urls_do_sitemap()
except requests.RequestException as erro:
    print("sem rede agora para consultar o sitemap.")
    print(type(erro).__name__, erro)
else:
    com_lastmod = [(u, m) for u, m in achados if m]
    mudaram = [u for u, m in com_lastmod if m[:10] > "2026-09-05"]

    print(f"URLs no sitemap agora        : {len(achados):,}")
    print(f"com <lastmod>                : {len(com_lastmod):,}")
    print(f"alteradas desde a coleta     : {len(mudaram):,}")


URLs no sitemap agora        : 3,498
com <lastmod>                : 3,498
alteradas desde a coleta     : 187
